# Step 02 — Feature Engineering

This notebook explores the engineered feature matrix produced by `pipelines/02_features.py`.
Features are derived from the raw macro series through:

1. Cross-asset ratios (10 derived columns)
2. Log transforms
3. Column selection (`initial_features`)
4. Bernstein polynomial gap-filling + Taylor extrapolation
5. Smoothed first/second/third derivatives via `np.gradient`
6. Final column selection (`clustering_features`)

**Run `python pipelines/02_features.py` before executing this notebook.**

## Setup & Imports

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "../src")
import logging
import pandas as pd
import matplotlib.pyplot as plt
from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import DATA_DIR, plotting
setup_logging("INFO")
log = logging.getLogger("02_features")
cfg = load()
run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)

In [ ]:
import subprocess
from pathlib import Path

def run_step_if_needed(step: int, required_paths: list, auto_run: bool = True) -> bool:
    """Run the pipeline step if any required output files are missing."""
    missing = [p for p in required_paths if not Path(p).exists()]
    if not missing:
        return True
    print(f"Missing: {[str(p) for p in missing]}")
    scripts = sorted(Path("../pipelines").glob(f"{step:02d}_*.py"))
    if not scripts:
        print(f"No pipeline script found for step {step}.")
        return False
    script = scripts[0]
    if not auto_run:
        print(f"  → Run: python {script}")
        return False
    print(f"  → Running {script.name} ...")
    result = subprocess.run(["python", str(script)], capture_output=True, text=True, cwd="..")
    out = result.stdout
    if len(out) > 4000:
        out = out[:2000] + "\n...\n" + out[-2000:]
    print(out)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
        return False
    print(f"  ✓ Step {step} complete.")
    return True

# Run feature engineering step if processed features are missing
run_step_if_needed(2, [DATA_DIR / "processed" / "features.parquet"])

## Load Feature Matrix

The processed feature matrix is stored at `data/processed/features.parquet`.
Each row is a quarter; columns include raw series, log-transforms, and derivatives.

In [2]:
FEATURES_PATH = DATA_DIR / "processed" / "features.parquet"

try:
    features = pd.read_parquet(FEATURES_PATH)
    log.info("Loaded features: shape=%s", features.shape)
except FileNotFoundError:
    print(f"ERROR: Feature file not found at {FEATURES_PATH}")
    print("Run 'python pipelines/02_features.py' first.")
    features = None
except Exception as exc:
    print(f"ERROR loading features: {exc}")
    features = None

2026-03-10 13:42:41 | INFO     | 02_features | Loaded features: shape=(305, 71)


## Column Groups by Prefix

Features are organized by prefix:
- **Raw / derived**: `sp500`, `credit_spread`, `div_yield2`, etc.
- **Log-transformed**: columns beginning with `log_`
- **First derivative**: columns ending with `_d1`
- **Second derivative**: columns ending with `_d2`
- **Third derivative**: columns ending with `_d3`

In [3]:
if features is not None:
    cols = features.columns.tolist()

    log_cols = [c for c in cols if c.startswith("log_")]
    d1_cols  = [c for c in cols if c.endswith("_d1")]
    d2_cols  = [c for c in cols if c.endswith("_d2")]
    d3_cols  = [c for c in cols if c.endswith("_d3")]
    raw_cols = [c for c in cols if c not in log_cols + d1_cols + d2_cols + d3_cols]

    print(f"Shape: {features.shape[0]} quarters x {features.shape[1]} columns")
    print(f"Date range: {features.index.min().date()} to {features.index.max().date()}")
    print()
    print(f"Raw / derived columns  : {len(raw_cols):3d}")
    print(f"Log-transformed (log_) : {len(log_cols):3d}")
    print(f"First derivative (_d1) : {len(d1_cols):3d}")
    print(f"Second derivative (_d2): {len(d2_cols):3d}")
    print(f"Third derivative (_d3) : {len(d3_cols):3d}")
    print(f"Total                  : {len(cols):3d}")
    print()
    print("Raw/derived sample:", raw_cols[:8])
    print("Log sample:",         log_cols[:8])
    print("d1 sample:",          d1_cols[:8])

Shape: 305 quarters x 71 columns
Date range: 1950-03-31 to 2026-03-31

Raw / derived columns  :   6
Log-transformed (log_) :  46
First derivative (_d1) :  34
Second derivative (_d2):  29
Third derivative (_d3) :   2
Total                  :  71

Raw/derived sample: ['credit_spread', 'gdp_growth', 'real_gdp_growth', 'sp500_pe', 'us_infl', 'market_code']
Log sample: ['log_cape_shiller_d1', 'log_cape_shiller_d2', 'log_cpi_d1', 'log_cpi_d2', 'log_div_yield_d1', 'log_div_yield_d2', 'log_div_yield2_d1', 'log_div_yield2_d2']
d1 sample: ['10yr_ustreas_d1', 'credit_spread_d1', 'div_minus_baa_d1', 'fred_aaa_d1', 'fred_baa_d1', 'fred_gs10_d1', 'fred_tb3ms_d1', 'gdp_growth_d1']


## Feature Distributions

Histograms for a subset of features. Higher-order derivatives (`_d2`, `_d3`) are
excluded by default to keep the grid readable. Useful for spotting outliers or
heavily skewed distributions before clustering.

In [4]:
if features is not None:
    plotting.plot_feature_distributions(features, run_cfg)

2026-03-10 13:42:50 | INFO     | market_regime.plotting | Saved plot: /Users/glestryc/personal/github_repos/claude-scratch-work/notebooks/../outputs/plots/02_feature_distributions.png
/Users/glestryc/personal/github_repos/claude-scratch-work/notebooks/../src/market_regime/plotting.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Feature Correlations

Correlation heatmap for the top-40 most-variable features.
Highly correlated clusters confirm expected economic relationships
(e.g., equity valuation metrics move together).

In [5]:
if features is not None:
    plotting.plot_feature_correlations(features, run_cfg, top_n=40)

2026-03-10 13:42:55 | INFO     | market_regime.plotting | Saved plot: /Users/glestryc/personal/github_repos/claude-scratch-work/notebooks/../outputs/plots/02_feature_correlations.png
/Users/glestryc/personal/github_repos/claude-scratch-work/notebooks/../src/market_regime/plotting.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## NaN Counts per Column

After Bernstein polynomial gap-filling, most interior NaNs should be resolved.
Remaining NaNs typically appear at the leading or trailing edges of the date range
where a series was not yet available.

In [6]:
if features is not None:
    nan_counts = features.isna().sum().sort_values(ascending=False)
    non_zero = nan_counts[nan_counts > 0]
    if non_zero.empty:
        print("No NaNs found in feature matrix — gap filling was complete.")
    else:
        print(f"Columns with NaN values ({len(non_zero)} of {len(features.columns)}):")
        display(non_zero.rename("nan_count").to_frame())

Columns with NaN values (66 of 71):


,nan_count
10yr_ustreas_d1,1
10yr_ustreas_d2,1
credit_spread_d1,1
div_minus_baa_d2,1
div_minus_baa_d1,1
...,...
log_sp500_adj_d3,1
real_gdp_growth_d1,1
real_gdp_per_cap_d2,1
real_gdp_per_cap_d1,1


## Describe Statistics — Top 20 by Std

Summary statistics sorted by standard deviation. The most variable features
are often the most informative for clustering.

In [7]:
if features is not None:
    desc = features.describe().T.sort_values("std", ascending=False)
    print("Top 20 features by standard deviation:")
    display(desc.head(20))

Top 20 features by standard deviation:


,count,mean,std,min,25%,50%,75%,max
sp500_pe,305.0,18.907057,11.389285,6.850000,12.840000,17.680000,22.250000,123.320000
real_gdp_per_cap_d1,304.0,1.956634,1.776608,-4.591099,0.929407,2.107941,3.157721,8.204169
market_code,304.0,1.210526,1.285263,0.000000,0.000000,1.000000,2.000000,4.000000
gdp_growth,305.0,0.064701,0.034559,-0.067300,0.045900,0.060900,0.082200,0.196500
us_infl,305.0,0.035234,0.028073,-0.014300,0.017100,0.028500,0.044600,0.147600
real_gdp_growth,305.0,0.031786,0.026091,-0.074000,0.019800,0.030500,0.044800,0.133700
sp500_pe_d1,304.0,0.000727,0.021226,-0.138220,-0.003673,0.001188,0.005750,0.126158
real_gdp_per_cap_d2,304.0,0.000010,0.004791,-0.012179,-0.002620,0.000061,0.002160,0.022192
credit_spread,305.0,0.009505,0.004225,0.003400,0.006800,0.008500,0.011300,0.033800
fred_tb3ms_d1,304.0,0.000102,0.003237,-0.009760,-0.001715,0.000145,0.002024,0.008767
